# Modelos lineales con PyMC

En esta notebook vamos a trabajar con las puntuaciones de unos niños en una prueba de capacidad cognitiva. Nos interesa saber cuál es su puntuación promedio y qué podemos decir sobre la puntuación de un nuevo niño. Primero vamos a usar solamente las puntuaciones y después veremos qué cambia al incorporar información sobre sus madres.

Antes de comenzar, retomemos la implementación de modelos en PyMC y los diagnósticos de MCMC que vimos en la clase anterior. Queremos evaluar si las muestras representan adecuadamente al _posterior_ y si el modelo puede generar datos parecidos a los observados. Son dos preguntas diferentes y vamos a volver sobre ellas a medida que construyamos los modelos.

## Preparación

En Google Colab, ejecutá primero la celda de instalación. Si Colab pide reiniciar la sesión después de instalar, reiniciala y volvé a ejecutar desde el principio.

In [ ]:
# Instalación opcional en Google Colab
import sys

if "google.colab" in sys.modules:
    %pip install -q "pymc==6.4.0" "preliz==0.28.0" "arviz-base==1.3.1" "matplotlib==3.11.2" "nutpie==0.16.11"
    %pip install -q polars h5netcdf h5py graphviz

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import preliz as pz
import pymc as pm
import xarray as xr

from pymc.model.fgraph import clone_model

plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#0072B2", "#E69F00", "#009E73", "#CC79A7"])

az.rcParams["stats.ci_kind"] = "eti"
az.rcParams["stats.ci_prob"] = 0.95

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)

## Repaso: los tiros libres de Caleb

Retomemos los 17 aciertos de Caleb en 23 intentos. Al igual que en la clase anterior, suponemos que la probabilidad de encestar es constante y que los tiros son independientes dado $\theta$. Con estos supuestos, nuestro modelo se escribe de la siguiente manera.

$$
\begin{aligned}
Y\mid\theta &\sim \operatorname{Binomial}(23,\theta),\\
\theta &\sim \operatorname{Beta}(1,1).
\end{aligned}
$$

Veamos cómo implementarlo en PyMC y obtener muestras con cuatro cadenas de NUTS. Las muestras y la información del muestreo van a quedar organizadas en un `DataTree`, como vimos antes.

In [ ]:
with pm.Model() as modelo:
    n = pm.Data("n", 23)
    y = pm.Data("y", 17)
    theta = pm.Beta("theta", alpha=1, beta=1)
    pm.Binomial("aciertos", n=n, p=theta, observed=y)

modelo

In [ ]:
with modelo:
    idata = pm.sample(draws=1000, tune=1000, chains=4, random_seed=SEMILLA)

### Diagnósticos

Antes de interpretar el _posterior_, veamos cómo se comportaron las cadenas. En las _trazas_ buscamos que recorran regiones similares, sin tendencias ni estancamientos. La autocorrelación nos ayuda a ver cuánto persiste la dependencia entre las muestras.

También podemos mirar el resumen que nos devuelve ArviZ. Esperamos valores de $\hat R$ cercanos a 1 y, si alguno supera 1.01, corresponde revisar el muestreo y, probablemente, también el modelo. Por su parte, `ess_bulk` y `ess_tail` nos dan una idea del tamaño efectivo de muestra para el centro y las colas de la distribución. La cantidad que necesitamos va a depender de lo que queramos estimar y de la precisión con la que necesitemos hacerlo.

En cuanto al MCSE, nos interesa que el error de Monte Carlo sea pequeño para la pregunta que queremos responder. Al compararlo con el desvío del _posterior_ podemos ver cuánto representa ese error numérico en relación con la incertidumbre sobre el parámetro.

Finalmente, revisemos si hubo divergencias. Si aparecen, puede haber problemas de exploración aunque los demás diagnósticos parezcan buenos. Por eso conviene mirar toda esta información en conjunto antes de continuar con la inferencia.

In [ ]:
az.summary(idata, var_names="theta", kind="diagnostics")

In [ ]:
az.plot_trace(idata, var_names="theta");

In [ ]:
trazas = az.plot_trace_dist(idata, var_names="theta", compact=False)
trazas.add_legend("chain");

In [ ]:
idata["sample_stats"]["diverging"].sum("draw")

In [ ]:
az.summary(idata, var_names="theta", kind="stats")

In [ ]:
az.plot_dist(idata, var_names="theta");

### Inferencia y predicción

Con las muestras del _posterior_ podemos estimar la media y un IC central del 95% para la probabilidad de encestar. También podemos preguntarnos qué tan probable es que $\theta$ supere 0.6.

Para responder a esta última pregunta, calculamos la proporción de muestras que superan ese valor. Si después queremos probar con otro umbral, basta con cambiarlo y volver a hacer la cuenta.

In [ ]:
umbral_theta = 0.6
theta_muestras = idata["posterior"]["theta"]

Calculemos primero la media _a posteriori_.

In [ ]:
theta_muestras.mean().item()

Y luego los extremos de un IC central del 95%.

In [ ]:
theta_muestras.quantile([0.025, 0.975]).to_numpy()

Para estimar $P(\theta > 0.6 \mid y)$, veamos qué proporción de las muestras supera el umbral.

In [ ]:
(theta_muestras > umbral_theta).mean().item()

Hasta acá hicimos inferencia sobre la probabilidad de encestar. Si queremos saber cuántos tiros podría acertar Caleb en un nuevo conjunto de intentos, también tenemos que considerar la variabilidad de esos tiros.

Como la simulación es sencilla, podemos usar una variable aleatoria de PreliZ directamente:

In [ ]:
n_nuevo = 20
umbral_aciertos = 17

In [ ]:
y_aciertos_pred = pz.Binomial(n=n_nuevo, p=theta_muestras).rvs()
y_aciertos_pred

In [ ]:
(y_aciertos_pred > umbral_aciertos).mean()

También podemos realizar la tarea usando herramientas propias de PyMC.

Para ello, se usa `pm.sample_posterior_predictive()`. Primero clonamos el modelo con `clone_model()` y cambiamos la cantidad de intentos mediante `pm.set_data()`. De esta forma podemos predecir para otra cantidad de tiros y conservar el modelo original. Al pasar `predictions=True`, las simulaciones se guardan en el grupo `predictions`.

In [ ]:
with clone_model(modelo) as modelo_caleb_pred:
    pm.set_data({"n": n_nuevo})
    pred_caleb = pm.sample_posterior_predictive(
        idata, var_names=["aciertos"], predictions=True, random_seed=SEMILLA
    )

In [ ]:
pred_caleb["predictions"]["aciertos"]

In [ ]:
umbral_aciertos = 17
aciertos_nuevos = pred_caleb["predictions"]["aciertos"]
prob_aciertos = (aciertos_nuevos > umbral_aciertos).mean().item()
prob_aciertos

In [ ]:
conteos = np.arange(n_nuevo + 1)
masa = np.bincount(aciertos_nuevos.to_numpy().ravel(), minlength=n_nuevo + 1)
masa = masa / aciertos_nuevos.size

In [ ]:
colores = np.where(conteos > umbral_aciertos, "C1", "C0")

fig, ax = plt.subplots(figsize=(7, 4))
ax.vlines(conteos, 0, masa, color=colores, linewidth=2.4)
ax.scatter(conteos, masa, color=colores, s=35)
ax.axvline(umbral_aciertos + 0.5, color="C1", ls="--", label=f"Más de {umbral_aciertos} aciertos")
ax.set(xlabel=rf"$\tilde y \mid n = {n_nuevo}$", ylabel=r"$p(\tilde y \mid y)$")
ax.legend();

## Más diagnósticos: cuando el muestreo tiene problemas

ArviZ incluye muestras de dos ejecuciones de un modelo muy popular en la literatura bayesiana, conocido como "el modelo de las 8 escuelas". Este modelo busca comparar el efecto de un programa de preparación para exámenes entre 8 escuelas. Todavía no vamos a desarrollar ese modelo, pero podemos usarlo para comparar el comportamiento de las cadenas. Para leer las salidas alcanza con saber que `mu` representa una media común, `theta` representa la media de cada escuela y `tau` una escala de variación entre escuelas.

Ambos conjuntos de muestras corresponden al mismo modelo matemático, implementado con dos parametrizaciones distintas. En `centered_eight` el _sampler_ tiene dificultades para explorar el _posterior_, mientras que en `non_centered_eight` la exploración mejora.

In [ ]:
muestras_centradas = az.load_arviz_data("centered_eight")
muestras_no_centradas = az.load_arviz_data("non_centered_eight")

### Traceplots

Veamos primero las trazas y las densidades por cadena para la parametrización centrada. Buscamos que las cadenas recorran regiones similares y que sus densidades se superpongan.

In [ ]:
az.plot_trace_dist(muestras_centradas, var_names=["mu", "tau"], compact=False);

En la parametrización centrada se ven diferencias entre las densidades estimadas con cada cadena y tramos donde la exploración avanza lentamente. Estas señales nos hacen dudar de la mezcla. Comparemos con la parametrización no centrada.

In [ ]:
az.plot_trace_dist(muestras_no_centradas, var_names=["mu", "tau"], compact=False);

### ESS y $\hat R$

Con la parametrización no centrada las densidades de las cadenas se parecen más y las trazas recorren las mismas regiones con mayor facilidad. Veamos si el resumen numérico acompaña esta mejora.

In [ ]:
az.summary(muestras_centradas, var_names=["mu", "tau"], kind="diagnostics")

En el caso centrado, $\hat R$ vale aproximadamente 1.03 para ambos parámetros y supera la referencia de 1.01. Además, los tamaños efectivos son pequeños, sobre todo para `tau`, con `ess_bulk` de 127 y `ess_tail` de 214. El resumen refuerza las dificultades que vimos en las trazas. Veamos los mismos indicadores para el caso no centrado.

In [ ]:
az.summary(muestras_no_centradas, var_names=["mu", "tau"], kind="diagnostics")

Ahora $\hat R$ queda cerca de 1 y los tamaños efectivos aumentan: para `tau`, `ess_bulk` pasa a 833 y `ess_tail` a 659. El MCSE de la media también baja para ambos parámetros. Con estas muestras, la parametrización no centrada permite estimar las medias con menor error de Monte Carlo.

### Energía marginal y energía de transición

NUTS usa una energía que combina la posición en el espacio de parámetros con un _momentum_ auxiliar. `az.plot_energy()` compara la distribución de las energías visitadas (_marginal_) con la de los cambios de energía entre iteraciones consecutivas (_transition_).

**Buscamos que ambas densidades se vean lo más parecidas posible.** Si la marginal es más ancha que la de transición, los cambios entre iteraciones son pequeños frente al rango de energías que hay que recorrer. Esto indica una exploración lenta.

El panel de BFMI ofrece un resumen numérico de esta comparación. Veamos primero las densidades del caso centrado y después las del no centrado.

In [ ]:
az.plot_energy(muestras_centradas, figure_kwargs={"figsize": (8, 3)});

En la parametrización centrada, la densidad marginal es bastante más ancha que la de transición. Los cambios de energía entre iteraciones son pequeños frente al rango que hay que recorrer. Esto agrega otra señal de dificultad de exploración.

In [ ]:
az.plot_energy(muestras_no_centradas, figure_kwargs={"figsize": (8, 3)});

En la parametrización no centrada las dos densidades se parecen mucho más, tanto en su forma como en su ancho. Esto sugiere una exploración más eficiente y acompaña la mejora que vimos en los otros diagnósticos. Todavía necesitamos revisar las divergencias.

### Divergencias

Una de las ventajas del algoritmo NUTS es que "falla con estilo". Por ejemplo, cuando intenta transicionar de una región de baja curvatura a otra de alta curvatura, la trayectoria calculada numéricamente mediante el integrador _leapfrog_ puede diverger. Las divergencias advierten sobre regiones que podrían estar mal exploradas, incluso cuando otros diagnósticos parecen buenos.

Calculemos primero la cantidad de divergencias por cadena para la parametrización centrada.

In [ ]:
muestras_centradas["sample_stats"]["diverging"].sum("draw")

Hay 19 divergencias en total, repartidas entre las cuatro cadenas. Para ver dónde ocurren, visualizamos las muestras de `tau` junto con el efecto `theta` para la escuela Choate y marcamos las transiciones divergentes con otro color.

In [ ]:
az.plot_pair(
    muestras_centradas,
    var_names=["tau", "theta"],
    visuals={"divergence": True},  # Las divergencias se muestran con otro color.
    coords={"school": "Choate"},  # Seleccionamos una escuela.
    figure_kwargs={"figsize": (12, 6)},
);

Las divergencias aparecen principalmente donde `tau` es pequeño. Esa región presenta dificultades para el _sampler_, por lo que las muestras centradas podrían representar de manera inadecuada una parte del _posterior_. Comparemos el conteo con la parametrización no centrada.

In [ ]:
muestras_no_centradas["sample_stats"]["diverging"].sum("draw")

En la ejecución no centrada no hubo divergencias en ninguna cadena. Veamos también el gráfico para contrastar la región de valores pequeños de `tau`.

In [ ]:
az.plot_pair(
    muestras_no_centradas,
    var_names=["tau", "theta"],
    visuals={"divergence": True},
    coords={"school": "Choate"},
    figure_kwargs={"figsize": (12, 6)},
);

La parametrización no centrada permite explorar valores pequeños de `tau` sin las divergencias que vimos antes. La ausencia de divergencias acompaña la mejora de los otros diagnósticos, aunque por sí sola no garantiza que las muestras representen adecuadamente todo el _posterior_.

En este ejemplo, cambiar la parametrización mejora la exploración del mismo modelo matemático. Esto ayuda a entender por qué conviene revisar su implementación cuando aparecen estas dificultades. Si el muestreador las mantiene, pedir más iteraciones puede prolongar el problema.

**¿Qué hacer cuando los diagnósticos dan mal?** Al final de la notebook se muestran [algunas sugerencias](#que-hacer-diagnosticos).

Pasemos ahora a los modelos lineales para estudiar las puntuaciones de los niños e incorporar información sobre sus madres.

## Modelos lineales

### Puntuaciones de los niños

El conjunto de datos `kidiq` se utiliza en _Regression and Other Stories_, de Gelman, Hill y Vehtari. Cada fila contiene información de un niño y su madre, con las variables que se describen debajo.

| Variable | Descripción |
| --- | --- |
| `kid_score` | Puntuación del niño en una prueba de capacidad cognitiva. |
| `mom_iq` | Puntuación de CI de la madre. |
| `mom_hs` | Vale 1 si la madre terminó la secundaria y 0 si no la terminó. |
| `mom_work` | Situación laboral de la madre durante los primeros tres años del niño. |
| `mom_age` | Edad de la madre al momento del nacimiento. |

Vamos a trabajar con las tres primeras variables. Antes de proponer el modelo, conviene entender qué miden estas puntuaciones y en qué escala están expresadas.

Las puntuaciones de los niños corresponden al test de vocabulario receptivo Peabody, o PPVT-R. En esta prueba, el niño escucha una palabra y elige entre cuatro imágenes la que representa su significado. El protocolo revisado del NLSY permite registrar puntuaciones estandarizadas entre 20 y 160, asignando esos valores a los resultados que quedan por debajo o por encima de los límites de las tablas.

Por su parte, los tests de IQ suelen usar una escala con media 100 y desvío 15 en su población de referencia. Aproximadamente dos tercios de las personas se encuentran entre 85 y 115, y alrededor del 95% entre 70 y 130.

En primer lugar nos interesa estimar la puntuación promedio de los niños y la probabilidad de que un nuevo niño supere un valor determinado. Después veremos cómo cambian esas respuestas cuando conocemos el IQ de su madre y si terminó la secundaria.

Comencemos con un modelo sin covariables. Los rangos que vamos a usar para elegir sus _priors_ son supuestos para este ejemplo. En una aplicación necesitaríamos discutirlos con alguien que conozca la prueba y la población con la que estamos trabajando.

#### Un modelo sin covariables

Como primera aproximación, supongamos que las puntuaciones siguen una distribución normal con media y desvío desconocidos. Nuestro modelo queda de la siguiente manera:

$$
\begin{aligned}
Y_i\mid\mu,\sigma &\overset{\mathrm{iid}}\sim \operatorname{Normal}(\mu,\sigma^2),\\
\mu &\sim \operatorname{Normal}(m,s^2),\\
\sigma &\sim \operatorname{Gamma}(a,b).
\end{aligned}
$$

Acá $\mu$ representa la puntuación promedio de los niños y $\sigma$ la variabilidad entre ellos. Como desconocemos ambas cantidades, necesitamos un _prior_ para cada una. Vamos a suponer que son independientes _a priori_.

Para la media podemos usar una normal que asigne un 95% de probabilidad al intervalo de 70 a 130 puntos. Para el desvío necesitamos una distribución con soporte positivo, por lo que vamos a usar una gamma que concentre un 95% de probabilidad entre 0 y 40 puntos.

En la gamma, $a$ es el parámetro de forma y $b$ el de tasa.

Debajo usamos PreliZ para encontrar los parámetros que satisfacen los rangos que propusimos.

Vamos con el _prior_ para $\mu$:

In [ ]:
prior_mu = pz.maxent(pz.Normal(), lower=70, upper=130, mass=0.95, plot=False)
prior_mu.plot_pdf();

Y ahora el _prior_ para $\sigma$:

In [ ]:
prior_sigma = pz.maxent(pz.Gamma(), lower=0, upper=40, mass=0.95, plot=False)
prior_sigma.plot_pdf();

Para obtener estas distribuciones usamos `pz.maxent()`, que busca la distribución de mayor entropía dentro de la familia elegida y bajo la restricción que le pasamos. En nuestro caso, `lower` y `upper` indican los extremos del intervalo y `mass` la probabilidad que queremos asignarle.

La función ajusta la distribución recibida y devuelve ese mismo objeto. Después podemos recuperar sus parámetros con `.params` (o el nombre del parámetro) y usarlos en PyMC sin redondearlos.

Ahora bien, ¿qué puntuaciones podríamos observar bajo estos supuestos? Para averiguarlo, podemos tomar una media y un desvío de sus _priors_ y generar una puntuación normal con esos valores. Al repetir este proceso obtenemos muestras de la predictiva _a priori_, que podemos examinar antes de usar los datos.

In [ ]:
mu_prior_muestras = prior_mu.rvs(1000, random_state=1)
sigma_prior_muestras = prior_sigma.rvs(1000, random_state=2)
score_prior = rng.normal(mu_prior_muestras, sigma_prior_muestras)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(score_prior, bins=40, density=True, alpha=0.7)
ax.set(xlabel="Puntuación nueva", ylabel="Densidad");

Se puede ver que estos _priors_ permiten un rango amplio de puntuaciones. Esto sucede porque estamos usando una normal para aproximar la distribución de las puntuaciones y su soporte es toda la recta real.

La simulación nos permite ver cuánta probabilidad estamos asignando a valores poco plausibles. Si nos pareciera demasiada, sería un motivo para revisar los supuestos antes de continuar con los datos.

#### Carga y exploración de datos

Leemos el CSV directamente del [repositorio de _Regression and Other Stories_](https://github.com/avehtari/ROS-Examples/tree/master/KidIQ).

In [ ]:
url_datos = "https://raw.githubusercontent.com/avehtari/ROS-Examples/master/KidIQ/data/kidiq.csv"
df_kid = pl.read_csv(url_datos)
df_kid

In [ ]:
df_kid.null_count()

Tenemos 434 observaciones y no hay valores faltantes en las variables que vamos a usar. Empecemos por mirar solamente las puntuaciones de los niños.

In [ ]:
score = df_kid["kid_score"].to_numpy()
n_chicos = len(df_kid)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(score, bins=30, alpha=0.7)
ax.set(xlabel="Puntuación del niño", ylabel="Cantidad de niños");

#### Implementación en PyMC

En este primer modelo tenemos solamente dos parámetros escalares, $\mu$ y $\sigma$, que comparten todas las observaciones.

Guardamos las puntuaciones en `pm.Data()` para poder cambiar la cantidad de niños al predecir. Con `shape=score_data.shape`, la cantidad de puntuaciones que genera el modelo sigue el tamaño de ese vector.

In [ ]:
with pm.Model() as modelo_0:
    score_data = pm.Data("score_data", score)
    mu = pm.Normal("mu", mu=prior_mu.mu, sigma=prior_mu.sigma)
    sigma = pm.Gamma("sigma", alpha=prior_sigma.alpha, beta=prior_sigma.beta)
    pm.Normal("score", mu=mu, sigma=sigma, observed=score_data, shape=score_data.shape)

modelo_0.to_graphviz()

In [ ]:
with modelo_0:
    idata_0 = pm.sample(chains=4, random_seed=SEMILLA)

Primero, revisemos el muestreo.

In [ ]:
az.summary(idata_0, var_names=["mu", "sigma"], kind="diagnostics", round_to=3)

In [ ]:
idata_0["sample_stats"]["diverging"].sum("draw").to_numpy()

In [ ]:
az.plot_trace(idata_0);

Veamos también el _posterior_ conjunto de $\mu$ y $\sigma$ a través de sus muestras.

In [ ]:
mu_muestras_0 = idata_0["posterior"]["mu"].to_numpy().ravel()
sigma_muestras_0 = idata_0["posterior"]["sigma"].to_numpy().ravel()

fig, ax = plt.subplots()
ax.scatter(mu_muestras_0, sigma_muestras_0, alpha=0.7, s=10)
ax.set(xlabel=r"$\mu$ (puntos)", ylabel=r"$\sigma$ (puntos)");

La mayor densidad se concentra cerca de una media de 87 puntos para $\mu$ y una media de 20 puntos para $\sigma$. El gráfico permite ver la incertidumbre sobre ambas cantidades al mismo tiempo. Para responder solamente por la puntuación promedio, vamos a mirar la marginal de $\mu$.

#### ¿Cuál es la puntuación promedio?

Una vez revisado el muestreo, podemos usar las muestras de $\mu$ para responder la primera pregunta. Debajo calculamos la media _a posteriori_ y un IC central del 95% para la puntuación promedio de la población representada por esta muestra, bajo los supuestos del modelo.

In [ ]:
az.summary(idata_0, var_names="mu", kind="stats", ci_prob=0.95, round_to=2)

In [ ]:
az.plot_dist(idata_0, var_names="mu", ci_kind="eti", ci_prob=0.95);

La puntuación promedio se encuentra alrededor de 87 puntos, con un IC central del 95% que va aproximadamente de 85 a 89. Ese intervalo refleja la incertidumbre que tenemos sobre la puntuación promedio.

#### ¿Y un nuevo niño?

La puntuación de un niño puede estar bastante alejada del promedio. Por eso, para responder esta pregunta tenemos que incorporar tanto la incertidumbre sobre los parámetros como la variabilidad de las puntuaciones dado $(\mu,\sigma)$. La distribución predictiva _a posteriori_ reúne esas dos fuentes de incertidumbre.

$$
p(\tilde y\mid\boldsymbol y)=\int p(\tilde y\mid\mu,\sigma)\,p(\mu,\sigma\mid\boldsymbol y)\,d\mu\,d\sigma.
$$

En la práctica, aproximamos la distribución utilizando Monte Carlo.

Para obtener muestras, en este caso sencillo, podemos simular directamente desde una distribución normal en `PreliZ`.

In [ ]:
mu_muestras = idata_0["posterior"]["mu"].to_numpy()
sigma_muestras = idata_0["posterior"]["sigma"].to_numpy()

y_pred_muestras = pz.Normal(mu=mu_muestras, sigma=sigma_muestras).rvs(random_state=12)
y_pred_muestras

In [ ]:
umbral_score = 110
(y_pred_muestras > umbral_score).mean()

También es posible usar `pm.sample_posterior_predictive()`.

Para cada par $(\mu_s,\sigma_s)$ del _posterior_, PyMC genera una puntuación normal. Luego podemos estimar la probabilidad de superar un umbral calculando una proporción, como hicimos con los tiros libres.

Como queremos predecir para un solo niño, clonamos el modelo y usamos `pm.set_data()` para reemplazar `score_data` por un vector de longitud uno. El cero funciona como marcador del tamaño y su valor no interviene en la simulación. PyMC va a usar las muestras del _posterior_ que hay en `idata_0`, sin volver a ajustar el modelo.

Con `predictions=True`, las simulaciones quedan en un nuevo grupo llamado `predictions`.

Al trabajar sobre una copia del modelo, conservamos las puntuaciones originales para el chequeo predictivo que viene después.

In [ ]:
with clone_model(modelo_0) as modelo_0_pred:
    pm.set_data({"score_data": [0]})
    pred_0 = pm.sample_posterior_predictive(
        idata_0, var_names=["score"], predictions=True, random_seed=SEMILLA
    )

score_nuevo_0 = pred_0["predictions"]["score"].to_numpy().squeeze(axis=-1)

In [ ]:
umbral_score = 110
(score_nuevo_0 > umbral_score).mean()

Con un umbral de 110 puntos obtenemos una probabilidad cercana al 13%. Si queremos probar con otro valor, podemos cambiar `umbral_score` en su celda y volver a calcular la probabilidad. Como ya tenemos las simulaciones, podemos reutilizarlas para responder las nuevas preguntas.

Veamos también un intervalo predictivo central del 95% para la puntuación de un nuevo niño.

In [ ]:
np.quantile(score_nuevo_0, [0.025, 0.975])

In [ ]:
distribuciones_0 = xr.Dataset(
    {
        "Puntuación de un nuevo niño": (("chain", "draw"), score_nuevo_0),
        "Puntuación promedio": idata_0["posterior"]["mu"],
    }
)

az.plot_dist(
    distribuciones_0,
    kind="kde",
    ci_kind="eti",
    ci_prob=0.95,
    col_wrap=2,
    figure_kwargs={"figsize": (10, 3.5), "sharex": True},
);

Podemos notar que el intervalo predictivo para un niño nuevo es mucho más amplio que el IC de la media. Esto se debe a que, además de la incertidumbre sobre el promedio, estamos considerando la variabilidad entre niños.

#### ¿Qué tan bien describe los datos?

Una forma de evaluar la adecuación del modelo es verificando si puede generar conjuntos de puntuaciones parecidos al que observamos.

Para eso usamos `pm.sample_posterior_predictive()`, que simula un conjunto con la misma cantidad de observaciones que la muestra para cada extracción del _posterior_. Con `extend_inferencedata=True`, esos conjuntos quedan guardados en el grupo `posterior_predictive`.

Después podemos compararlos con los datos mediante `az.plot_ppc_dist()`. En el gráfico, las curvas azules corresponden a los conjuntos simulados y la negra a las observaciones. Esta comparación es un **chequeo predictivo _a posteriori_** y nos ayuda a reconocer qué aspectos de los datos está reproduciendo el modelo.

In [ ]:
with modelo_0:
    pm.sample_posterior_predictive(
        idata_0, var_names=["score"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_0, var_names="score", num_samples=100);

Las réplicas normales son simétricas y permiten comparar el centro, la dispersión y las colas con los datos. Aun si esta comparación global fuera satisfactoria, todavía podríamos estar dejando afuera una relación importante. Por ejemplo, ¿qué pasa si la puntuación media cambia con el IQ de la madre?

### Regresión lineal con IQ materno

Hasta acá usamos la misma media para todos los niños. Ahora vamos a permitir que esa media dependa del IQ de su madre.

Antes, hagamos un rapidísimo repaso por el modelo de regresión lineal.

Supongamos que contamos con un conjunto de observaciones a los cuales se les mide el par $(X, Y)$:

<img src="figuras/regresion_dispersion.png" width="700px">

En un modelo de regresión lineal suponemos que la media de la respuesta $Y$ es una función lineal de la predictora $X$.

Escribimos: $\mathbb{E}(Y \mid X = x) = \mu(x) = \beta_0 + \beta_1 x$.

- $\beta_0$ es la media de $Y$ cuando $X=0$
- $\beta_1$ es el cambio en la media de $Y$ al aumentar $X$ en una unidad.

Como no conocemos los valores de $\beta_0$ y $\beta_1$, representamos la incertidumbre con una distribución de probabilidad.
Esta se propaga a la recta de regresión y se representa en el panel derecho de la figura debajo.

<img src="figuras/regresion_media_incertidumbre.png" width="1100px">

Ahora agreguemos las observaciones y conservemos la banda que representa la incertidumbre sobre la recta. Muchos puntos quedan fuera de ella porque, para un mismo valor de $X$, las observaciones también varían alrededor de la media $\mu(x)$.

Para representar esa variabilidad, completamos el modelo con una distribución normal para $Y\mid X=x$. En el panel derecho dibujamos esas distribuciones para tres valores fijos de $X$.

El parámetro $\sigma$ indica cuánto se dispersan las observaciones en la dirección de $Y$. **Cuanto mayor es $\sigma$, más alejados de $\mu(x)$ pueden estar los valores de $Y$**. Como usamos el mismo $\sigma$ para todo $X$, las tres normales tienen la misma dispersión. La flecha muestra una distancia proporcional a $\sigma$.

Así distinguimos la incertidumbre sobre la media condicional, representada por la banda azul, de la variabilidad de una observación individual alrededor de esa media.

<img src="figuras/regresion_observaciones_sigma.png" width="1100px">

Volviendo a las puntuaciones...

En este modelo de regresión lineal suponemos que la puntuación media cambia con el IQ materno siguiendo una recta. Si llamamos $x_i$ al IQ de la madre del niño $i$, podemos escribir el modelo de la siguiente manera.

$$
\begin{aligned}
Y_i\mid \mu_i,\sigma &\sim \operatorname{Normal}(\mu_i,\sigma^2)\\
\mu_i &= \beta_0+\beta_1 x_i
\end{aligned}
$$

Esta representación es equivalente a escribir $Y_i=\beta_0+\beta_1 x_i+\varepsilon_i$, con errores normales independientes de media cero y varianza $\sigma^2$. Al trabajar bajo un enfoque bayesiano, además vamos a proponer distribuciones para las cantidades desconocidas.

Estamos suponiendo que la relación entre el IQ materno y la media de las puntuaciones es lineal. Para un IQ dado, las puntuaciones siguen una normal con desvío $\sigma$, y usamos ese mismo desvío para todos los valores de IQ. Este último supuesto se conoce como **homocedasticidad**. También suponemos que las observaciones son independientes dados los parámetros y las covariables.

Vale la pena detenerse en la normalidad que estamos suponiendo. Se refiere a **$Y\mid X$**, es decir, a las puntuaciones de niños para un IQ materno fijo. Si reunimos niños cuyas madres tienen distintos IQ, sus medias condicionales también pueden ser distintas y la distribución de todas las puntuaciones juntas puede tener otra forma.

#### _Priors_ para la regresión

Ahora tenemos que proponer _priors_ para los coeficientes de la recta $(\beta_0,\beta_1)$ y para el desvío residual $\sigma$. Vamos a suponer que son independientes _a priori_. Antes de elegirlos, pensemos qué representa cada parámetro.

El intercepto $\beta_0$ es la puntuación media para un IQ materno igual a cero. Ese valor queda muy lejos de los IQ observados y resulta difícil darle una interpretación práctica. Como primera aproximación, usamos una normal que asigne un 95% de probabilidad al intervalo de 0 a 100 puntos. Este es un supuesto sobre la recta extrapolada hasta IQ cero.

Para la pendiente $\beta_1$ podemos preguntarnos cuánto podría cambiar la puntuación media de un niño por cada punto adicional de IQ de su madre. Elegimos una normal centrada en cero, con un 95% de probabilidad entre -2 y 2. Este _prior_ asigna la misma probabilidad a pendientes positivas y negativas. Además, deja un 5% de probabilidad fuera de ese intervalo, de modo que $\pm2$ no son límites de los cambios posibles.

Finalmente, $\sigma$ representa la variabilidad de las puntuaciones para un IQ materno dado. Conservamos la $\operatorname{Gamma}(a,b)$ que usamos antes, con un 95% de probabilidad entre 0 y 40 puntos. Los rangos elegidos son supuestos para este ejemplo. En una aplicación convendría discutirlos con alguien que conozca el problema; también se podría usar un chequeo predictivo _a priori_ para evaluar las puntuaciones que permiten generar.

In [ ]:
prior_beta_0 = pz.maxent(pz.Normal(), lower=0, upper=100, mass=0.95, plot=False)
prior_beta_0.plot_pdf();

Ahora hagamos lo mismo con la pendiente.

In [ ]:
prior_beta_1 = pz.maxent(pz.Normal(), lower=-2, upper=2, mass=0.95, plot=False)
prior_beta_1.plot_pdf();

Visualicemos los datos:

In [ ]:
iq_madre = df_kid["mom_iq"].to_numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(iq_madre, score, s=18, alpha=0.5)
ax.set(xlabel="IQ de la madre", ylabel="Puntuación del niño");

Las madres con mayor IQ tienden a tener niños con puntuaciones mayores. Sin embargo, para un IQ materno dado hay bastante variabilidad en la puntuación de los niños.

#### Implementación y diagnósticos

Este modelo tiene tres parámetros libres, $\beta_0$, $\beta_1$ y $\sigma$.

Los $\mu_i$ también son parámetros, pero no libres, ya que se calculan a partir de los coeficientes y los IQ conocidos. Su _prior_ queda determinado por el _prior_ sobre $\beta_0$ y $\beta_1$ y los IQ.

En la implementación vamos a usar `pm.Data()` para registrar el IQ materno y `pm.Deterministic()` para guardar las medias condicionales. Ambas cantidades tienen una entrada por observación. Al registrar la covariable de esta forma, después podemos cambiar sus valores para obtener predicciones para otros IQ sin volver a ajustar el modelo.

En esta regresión usamos `obs_idx` para identificar a los niños en las salidas. Sus valores se guardan en `coords` y `dims="obs_idx"` asocia los vectores a esa dimensión.

In [ ]:
coords = {"obs_idx": np.arange(n_chicos)}

with pm.Model(coords=coords) as modelo_1:
    iq_data = pm.Data("iq_madre", iq_madre, dims="obs_idx")
    beta_0 = pm.Normal("beta_0", mu=prior_beta_0.mu, sigma=prior_beta_0.sigma)
    beta_1 = pm.Normal("beta_1", mu=prior_beta_1.mu, sigma=prior_beta_1.sigma)
    sigma = pm.Gamma("sigma", alpha=prior_sigma.alpha, beta=prior_sigma.beta)

    mu = pm.Deterministic("mu", beta_0 + beta_1 * iq_data, dims="obs_idx")
    pm.Normal("score", mu=mu, sigma=sigma, observed=score, dims="obs_idx")

modelo_1.to_graphviz()

In [ ]:
with modelo_1:
    idata_1 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
parametros_regresion = ["beta_0", "beta_1", "sigma"]
az.summary(idata_1, var_names=parametros_regresion, kind="diagnostics", round_to=3)

In [ ]:
idata_1["sample_stats"]["diverging"].sum("draw")

In [ ]:
az.plot_trace_dist(idata_1, var_names=parametros_regresion);

Este modelo se podría escribir de una forma más eficiente. En la [sección adicional](#dependencia-intercepto-pendiente) exploramos cómo hacerlo centrando el IQ.

In [ ]:
az.summary(
    idata_1, var_names=parametros_regresion, kind="stats", ci_kind="eti", ci_prob=0.95, round_to=3
)

La pendiente se encuentra alrededor de 0.59, con un IC del 95% que va aproximadamente de 0.48 a 0.70. Esto significa que diez puntos adicionales de IQ materno se asocian con unos 5.9 puntos más en la puntuación media de los niños. Por su parte, el desvío residual queda alrededor de 18 puntos, frente a los aproximadamente 20 del modelo sin covariables.

Al interpretar estos resultados conviene recordar que $\sigma$ representa la dispersión alrededor de la media condicional. El intercepto, en cambio, corresponde a un IQ materno igual a cero, por lo que su interpretación implica extrapolar muy lejos de las madres observadas.

#### Medias y predicciones con PyMC

En lugar de interpretar el intercepto, podemos calcular la puntuación media para valores de IQ que tengan sentido en este problema. También nos interesa la puntuación de un nuevo niño y la probabilidad de que supere un umbral. Vamos a resolver estas consultas con PyMC para madres con IQ de 80, 100 y 120.

Con `pm.set_data()` cambiamos los valores de `iq_madre` y actualizamos las coordenadas de `obs_idx`. Como ahora tenemos tres registros, sus índices son 0, 1 y 2, en el mismo orden que los IQ de la lista. Usamos una copia del modelo con `clone_model()` para conservar los datos observados en el modelo original.

Al llamar a `pm.sample_posterior_predictive()` pedimos tanto `mu` como `score`. La primera variable contiene las medias condicionales y la segunda, puntuaciones de nuevos niños que incluyen la variabilidad representada por $\sigma$. Usamos las muestras del _posterior_ que ya obtuvimos, sin volver a ajustar el modelo.


In [ ]:
iq_nuevos = [80, 100, 120]
idx_nuevos = np.arange(len(iq_nuevos))

with clone_model(modelo_1):
    pm.set_data({"iq_madre": iq_nuevos}, coords={"obs_idx": idx_nuevos})
    preds_1 = pm.sample_posterior_predictive(
        idata_1, var_names=["mu", "score"], predictions=True, random_seed=SEMILLA
    )

In [ ]:
mu_preds_1 = preds_1["predictions"]["mu"]

PyMC devuelve las muestras como `DataArray` de xarray, con dimensiones `chain`, `draw` y `obs_idx`. Podemos operar directamente sobre ellas: al promediar sobre `chain` y `draw`, queda una media para cada consulta. Los índices 0, 1 y 2 corresponden a IQ de 80, 100 y 120, respectivamente.


In [ ]:
az.summary(mu_preds_1, round_to=2, kind="stats")

**¿Cuál es la probabilidad de que un niño supere 110 puntos?** Para esta pregunta usamos las puntuaciones predictivas guardadas en `score`. La comparación con el umbral produce valores verdaderos y falsos. Al promediarlos sobre `chain` y `draw`, obtenemos una probabilidad para cada IQ.

Para probar otros IQ, podemos cambiar `iq_nuevos`, volver a generar las predicciones y repetir las consultas. Si queremos cambiar solamente el umbral, alcanza con ejecutar la siguiente celda.


In [ ]:
umbral_score = 110
(preds_1["predictions"]["score"] > umbral_score).mean(("chain", "draw"))

Al aumentar el IQ materno, la distribución predictiva se desplaza hacia la derecha y cambia la probabilidad de superar el mismo umbral.

#### Visualizar las rectas y la incertidumbre

Podemos usar el mismo procedimiento con una grilla de IQ para visualizar las rectas y las bandas de incertidumbre. Estamos eligiendo valores donde evaluar el modelo, sin suponer que esos valores sean igualmente frecuentes en la población.


In [ ]:
iq_grid = np.linspace(70, 140, 100)
idx_grid = np.arange(len(iq_grid))

with clone_model(modelo_1) as modelo_1_pred:
    pm.set_data({"iq_madre": iq_grid}, coords={"obs_idx": idx_grid})
    pred_1 = pm.sample_posterior_predictive(
        idata_1, var_names=["mu", "score"], predictions=True, random_seed=SEMILLA
    )

In [ ]:
pred_1["predictions"]

Cada extracción de los coeficientes define una recta. Seleccionemos veinte al azar con `az.extract()` para ver la incertidumbre sobre la media condicional.

In [ ]:
rectas_1 = az.extract(
    pred_1, group="predictions", var_names="mu", num_samples=20, random_seed=SEMILLA
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for recta in rectas_1.transpose("sample", "obs_idx").to_numpy():
    ax.plot(iq_grid, recta, color="C0", alpha=0.4, lw=1)
ax.set(xlabel="IQ de la madre", ylabel="Puntuación media de los niños");

Con todas las muestras calculamos un IC central del 95% para la media en cada IQ y un intervalo predictivo del 95% para un niño. Superpongamos también las observaciones.

In [ ]:
mu_grid_1 = pred_1["predictions"]["mu"]
score_grid_1 = pred_1["predictions"]["score"]
mu_media_1 = mu_grid_1.mean(("chain", "draw")).to_numpy()
mu_inferior_1, mu_superior_1 = mu_grid_1.quantile([0.025, 0.975], dim=("chain", "draw")).to_numpy()
y_inferior_1, y_superior_1 = score_grid_1.quantile([0.025, 0.975], dim=("chain", "draw")).to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(iq_madre, score, s=12, alpha=0.4, color="0.4")
ax.fill_between(
    iq_grid, y_inferior_1, y_superior_1, color="C0", alpha=0.15, label="IC predictivo del 95%"
)
ax.fill_between(
    iq_grid,
    mu_inferior_1,
    mu_superior_1,
    color="C0",
    alpha=0.5,
    label="IC de la media del 95%",
)
ax.plot(iq_grid, mu_media_1, color="C0")
ax.set(xlabel="IQ de la madre", ylabel="Puntuación del niño")
ax.legend();

Podemos observar que la banda predictiva es bastante más ancha que la de la media. Esto indica que buena parte de la incertidumbre sobre un niño se debe a la variabilidad representada por $\sigma$, aun cuando tengamos bastante información sobre la media condicional.

Los intervalos se calculan para cada IQ por separado. Por eso, la banda de la media no implica que el 95% de las rectas completas quede dentro de ella. En cuanto a la banda predictiva, al trabajar con intervalos del 95% tampoco esperamos que contenga a todas las observaciones.


#### Revisar el ajuste

Además de comparar las distribuciones generadas para las puntuaciones, podemos mirar los residuos contra el IQ materno. Si encontramos un patrón sistemático, podría ser una señal de que la recta está dejando afuera parte de la asociación.

Los residuos se calculan como $y_i-\widehat\mu_i$, donde $\widehat\mu_i$ es la media _a posteriori_ de $\mu_i$. También conviene observar si su dispersión cambia con el IQ, ya que eso nos llevaría a revisar el supuesto de homocedasticidad.

In [ ]:
with modelo_1:
    pm.sample_posterior_predictive(
        idata_1, var_names=["score"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_1, var_names="score", num_samples=50);

In [ ]:
mu_ajustada_1 = idata_1["posterior"]["mu"].mean(("chain", "draw")).to_numpy()
residuos_1 = score - mu_ajustada_1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(iq_madre, residuos_1, s=15, alpha=0.5)
ax.axhline(0, color="0.4", ls="--")
ax.set(xlabel="IQ de la madre", ylabel="Residuo (puntos)");

### Incorporar una covariable binaria

Hasta acá usamos la misma recta para todos los niños. Pero también sabemos si la madre terminó la secundaria. **¿Podemos aprovechar esa información?**

Primero veamos cuántas observaciones tenemos para cada grupo y cómo se distribuyen en el gráfico de dispersión.

In [ ]:
secundaria_idx = df_kid["mom_hs"].to_numpy()
grupos_secundaria = ["No terminó", "Terminó"]
df_kid.group_by("mom_hs").len().sort("mom_hs")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
grupos = [0, 1]

for grupo, etiqueta in zip(grupos, grupos_secundaria):
    seleccion = secundaria_idx == grupo
    ax.scatter(iq_madre[seleccion], score[seleccion], s=20, alpha=0.6, label=etiqueta)

ax.set(xlabel="IQ de la madre", ylabel="Puntuación del niño")
ax.legend(title="Secundario de la madre");

Tenemos 93 madres que no terminaron la secundaria y 341 que sí. En el gráfico también podemos ver que las distribuciones de IQ son diferentes entre los grupos. Por eso, si comparamos directamente sus puntuaciones promedio, estamos reuniendo niños cuyas madres tienen distintos IQ.

Una forma de incorporar esta información es usar un intercepto para cada grupo y conservar una pendiente común. Después podemos extender el modelo para que la pendiente también sea diferente. En ambos casos vamos a seguir usando un desvío residual común.

#### Interceptos diferentes

Con un intercepto para cada grupo, el modelo queda de la siguiente manera.

$$
\begin{aligned}
Y_i\mid \mu_i, \sigma &\sim \text{Normal}(\mu_i,\sigma^2)\\
\mu_i &= \beta_{0,g[i]}+\beta_1x_i\\
\beta_{0,g} &\sim \text{Normal}(m_{\beta_0},s_{\beta_0}^2)\qquad g=0,1\\
\beta_1 &\sim \text{Normal}(m_{\beta_1},s_{\beta_1}^2)\\
\sigma &\sim \text{Gamma}(a, b)
\end{aligned}
$$

Acá $g[i]$ indica a qué grupo pertenece la madre de la observación $i$. Como la pendiente es común, las dos rectas son paralelas y la diferencia de medias entre grupos, para un mismo IQ, es siempre $\beta_{0,1}-\beta_{0,0}$.

Para cada intercepto vamos a conservar el _prior_ normal que usamos antes. Les asignamos _priors_ independientes con los mismos parámetros, de modo que las medias de los _priors_ no favorezcan una diferencia entre grupos. También conservamos los _priors_ de la pendiente y del desvío residual.

En PyMC, `beta_0` pasa a ser un vector con dos entradas. Con `beta_0[hs_idx_data]` seleccionamos la que corresponde a cada observación. Además, agregamos una dimensión `secundaria` con los nombres de los grupos para poder reconocerlos en las salidas.

In [ ]:
coords_grupos = {
    "obs_idx": np.arange(n_chicos),
    "secundaria": grupos_secundaria
}

with pm.Model(coords=coords_grupos) as modelo_2:
    iq_data = pm.Data("iq_madre", iq_madre, dims="obs_idx")
    hs_idx_data = pm.Data("hs_idx", secundaria_idx, dims="obs_idx")

    beta_0 = pm.Normal("beta_0", mu=prior_beta_0.mu, sigma=prior_beta_0.sigma, dims="secundaria")
    beta_1 = pm.Normal("beta_1", mu=prior_beta_1.mu, sigma=prior_beta_1.sigma)
    sigma = pm.Gamma("sigma", alpha=prior_sigma.alpha, beta=prior_sigma.beta)

    mu = pm.Deterministic("mu", beta_0[hs_idx_data] + beta_1 * iq_data, dims="obs_idx")
    pm.Normal("score", mu=mu, sigma=sigma, observed=score, dims="obs_idx")

modelo_2.to_graphviz()

In [ ]:
with modelo_2:
    idata_2 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_2, var_names=parametros_regresion, kind="diagnostics", round_to=3)

In [ ]:
idata_2["sample_stats"]["diverging"].sum("draw")

In [ ]:
az.plot_trace_dist(idata_2, var_names=parametros_regresion);

In [ ]:
az.summary(
    idata_2, var_names=parametros_regresion, kind="stats", ci_kind="eti", ci_prob=0.95, round_to=3
)

Para estimar la diferencia entre grupos usamos las dos entradas de `beta_0` de cada extracción. Es una diferencia **a igual IQ**, bajo el supuesto de rectas paralelas.

In [ ]:
interceptos_2 = idata_2["posterior"]["beta_0"]
diferencia_2 = interceptos_2.sel(secundaria="Terminó") - interceptos_2.sel(secundaria="No terminó")

In [ ]:
fig, ax = plt.subplots()
ax.hist(diferencia_2.to_numpy().flatten(), bins=50, density=True)
ax.set(
    xlabel=r"$\beta_{0, \text{Terminó}} - \beta_{0, \text{No terminó}}$",
    ylabel="Densidad"
);

In [ ]:
diferencia_2.mean().item()

In [ ]:
diferencia_2.quantile([0.025, 0.975]).to_numpy()

In [ ]:
(diferencia_2 > 0).mean().item()

Según este modelo, la puntuación media de los niños cuyas madres terminaron la secundaria se encuentra unos 6 puntos por encima de la del otro grupo, a igual IQ materno. El IC del 95% para esa diferencia va aproximadamente de 1.9 a 10.2 puntos.

Veamos cómo se ven las predicciones de ambos grupos usando la misma grilla de IQ. Con `np.tile()` repetimos esa grilla dos veces y con `np.repeat()` construimos el índice que identifica a cada grupo. Primero quedan todas las observaciones del grupo 0 y después las del grupo 1.

In [ ]:
iq_grid_grupos = np.tile(iq_grid, 2)
secundaria_grid = np.repeat([0, 1], len(iq_grid))
idx_grid = np.arange(len(iq_grid_grupos))

with clone_model(modelo_2):
    pm.set_data(
        {"iq_madre": iq_grid_grupos, "hs_idx": secundaria_grid},
        coords={"obs_idx": idx_grid},
    )
    pred_2 = pm.sample_posterior_predictive(
        idata_2, var_names=["mu", "score"], predictions=True, random_seed=SEMILLA + 9
    )

In [ ]:
mu_media_2 = pred_2["predictions"]["mu"].mean(("chain", "draw")).to_numpy()
mu_inferior_2, mu_superior_2 = (
    pred_2["predictions"]["mu"].quantile([0.025, 0.975], dim=("chain", "draw")).to_numpy()
)
y_inferior_2, y_superior_2 = (
    pred_2["predictions"]["score"].quantile([0.025, 0.975], dim=("chain", "draw")).to_numpy()
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True, layout="constrained")
grupos = [0, 1]

for ax, grupo, etiqueta in zip(axes, grupos, grupos_secundaria):
    tramo = slice(grupo * len(iq_grid), (grupo + 1) * len(iq_grid))
    seleccion = secundaria_idx == grupo
    ax.scatter(iq_madre[seleccion], score[seleccion], s=12, alpha=0.4, color="0.4")
    ax.fill_between(iq_grid, y_inferior_2[tramo], y_superior_2[tramo], color="C0", alpha=0.15)
    ax.fill_between(iq_grid, mu_inferior_2[tramo], mu_superior_2[tramo], color="C0", alpha=0.45)
    ax.plot(iq_grid, mu_media_2[tramo], color="C0")
    ax.set(title=etiqueta, xlabel="IQ de la madre")

axes[0].set_ylabel("Puntuación del niño");

#### Revisar el ajuste

Antes de cambiar las pendientes, generemos réplicas con los IQ y grupos observados. Comparemos su distribución con las puntuaciones de la muestra, como hicimos con los modelos anteriores.

In [ ]:
with modelo_2:
    pm.sample_posterior_predictive(
        idata_2, var_names=["score"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_2, var_names="score", num_samples=50);

En los gráficos de las predicciones podemos ver que las rectas son paralelas, como supusimos al escribir el modelo. Las bandas oscuras representan la incertidumbre sobre la media y las claras incorporan también la variabilidad de un niño nuevo.

Ahora bien, ¿tiene sentido suponer que la puntuación media cambia de la misma manera con el IQ en ambos grupos?

#### Interceptos y pendientes diferentes

Podemos extender el modelo anterior usando también una pendiente para cada grupo. El modelo queda de la siguiente manera.

$$
\begin{aligned}
Y_i\mid\mu_i,\sigma &\sim \text{Normal}(\mu_i,\sigma^2)\\
\mu_i &= \beta_{0,g[i]}+\beta_{1,g[i]}x_i\\
\beta_{0,g} &\sim \text{Normal}(m_{\beta_0},s_{\beta_0}^2)\qquad g=0,1\\
\beta_{1,g} &\sim \text{Normal}(m_{\beta_1},s_{\beta_1}^2)\qquad g=0,1\\
\sigma &\sim \text{Gamma}(a,b)
\end{aligned}
$$

Como antes, $g[i]$ indica a qué grupo pertenece la madre de la observación $i$. Ahora las rectas pueden tener pendientes diferentes, por lo que la diferencia de medias entre grupos puede cambiar con el IQ materno. Esto es lo que llamamos una **interacción** entre el IQ y secundaria.

Para cada intercepto y cada pendiente conservamos los _priors_ normales que usamos antes. En particular, el _prior_ de cada pendiente asigna un 95% de probabilidad al intervalo de -2 a 2. Los coeficientes son independientes _a priori_, con los mismos parámetros para ambos grupos. También conservamos el desvío residual común y su _prior_ $\operatorname{Gamma}(a,b)$.

En PyMC, tanto `beta_0` como `beta_1` son vectores con dimensión `secundaria`. Con `beta_0[hs_idx_data]` y `beta_1[hs_idx_data]` seleccionamos el intercepto y la pendiente que corresponden a cada observación. Usamos las mismas coordenadas `coords_grupos` y los mismos datos que en el modelo anterior. Los parámetros de los _priors_ siguen siendo fijos, por lo que todavía no estamos construyendo un modelo jerárquico.


In [ ]:
with pm.Model(coords=coords_grupos) as modelo_3:
    iq_data = pm.Data("iq_madre", iq_madre, dims="obs_idx")
    hs_idx_data = pm.Data("hs_idx", secundaria_idx, dims="obs_idx")

    beta_0 = pm.Normal("beta_0", mu=prior_beta_0.mu, sigma=prior_beta_0.sigma, dims="secundaria")
    beta_1 = pm.Normal("beta_1", mu=prior_beta_1.mu, sigma=prior_beta_1.sigma, dims="secundaria")
    sigma = pm.Gamma("sigma", alpha=prior_sigma.alpha, beta=prior_sigma.beta)

    mu = pm.Deterministic(
        "mu",
        beta_0[hs_idx_data] + beta_1[hs_idx_data] * iq_data,
        dims="obs_idx"
    )
    pm.Normal("score", mu=mu, sigma=sigma, observed=score, dims="obs_idx")

modelo_3.to_graphviz()

In [ ]:
with modelo_3:
    idata_3 = pm.sample(chains=4, random_seed=SEMILLA)

In [ ]:
az.summary(idata_3, var_names=parametros_regresion, kind="diagnostics", round_to=3)

In [ ]:
idata_3["sample_stats"]["diverging"].sum("draw")

In [ ]:
az.plot_trace_dist(idata_3, var_names=parametros_regresion);

In [ ]:
az.summary(
    idata_3, var_names=parametros_regresion, kind="stats", ci_kind="eti", ci_prob=0.95, round_to=3
)

Al permitir pendientes diferentes, la diferencia de medias entre grupos también cambia con el IQ. Podemos verlo al restar las dos rectas.

$$
\mu_1(x)-\mu_0(x)
= (\beta_{0,1}-\beta_{0,0})+(\beta_{1,1}-\beta_{1,0})x.
$$

La diferencia entre los interceptos sigue correspondiendo a IQ cero. Por eso, para interpretar la comparación entre grupos necesitamos evaluar las medias en valores de IQ que tengan sentido en este problema.

Debajo usamos la misma grilla que antes para visualizar las dos rectas.
Con `az.extract()` juntamos `chain` y `draw` en una dimensión `sample`. Después pasamos a NumPy con las muestras en las filas y los IQ en las columnas. Así calculamos medias e intervalos sobre `axis=0`.


In [ ]:
with clone_model(modelo_3):
    pm.set_data(
        {"iq_madre": iq_grid_grupos, "hs_idx": secundaria_grid},
        coords={"obs_idx": idx_grid},
    )
    pred_3 = pm.sample_posterior_predictive(
        idata_3, var_names=["mu", "score"], predictions=True, random_seed=SEMILLA + 11
    )

In [ ]:
mu_grid_3 = az.extract(pred_3, group="predictions", var_names="mu")
score_grid_3 = az.extract(pred_3, group="predictions", var_names="score")
mu_grid_3 = mu_grid_3.transpose("sample", "obs_idx").to_numpy()
score_grid_3 = score_grid_3.transpose("sample", "obs_idx").to_numpy()

mu_media_3 = mu_grid_3.mean(axis=0)
mu_inferior_3, mu_superior_3 = np.quantile(mu_grid_3, [0.025, 0.975], axis=0)
y_inferior_3, y_superior_3 = np.quantile(score_grid_3, [0.025, 0.975], axis=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True, layout="constrained")

for ax, grupo, etiqueta in zip(axes, grupos, grupos_secundaria):
    tramo = slice(grupo * len(iq_grid), (grupo + 1) * len(iq_grid))
    seleccion = secundaria_idx == grupo
    ax.scatter(iq_madre[seleccion], score[seleccion], s=20, alpha=0.4, color="0.4")
    ax.fill_between(iq_grid, y_inferior_3[tramo], y_superior_3[tramo], color="C0", alpha=0.15)
    ax.fill_between(iq_grid, mu_inferior_3[tramo], mu_superior_3[tramo], color="C0", alpha=0.45)
    ax.plot(iq_grid, mu_media_3[tramo])
    ax.set(title=etiqueta, xlabel="IQ de la madre")

axes[0].set_ylabel("Puntuación del niño");

Ahora las rectas pueden acercarse o alejarse según el IQ. También podemos notar que la incertidumbre aumenta en las regiones con pocas observaciones, especialmente para las madres que no terminaron la secundaria. Por eso conviene mirar el rango de datos de cada grupo al interpretar las comparaciones.

#### Volver a las preguntas

Veamos las puntuaciones medias y la probabilidad de superar un umbral para madres con IQ de 80, 100 y 120, en ambos grupos. Si queremos probar otros valores, podemos cambiar `iq_nuevos` y volver a simular usando el _posterior_ que ya obtuvimos. Para cambiar solamente el umbral, alcanza con modificarlo y volver a calcular la probabilidad.

In [ ]:
iq_nuevos = np.array([80, 100, 120])
iq_nuevos_grupos = np.tile(iq_nuevos, 2)
secundaria_nuevos = np.repeat([0, 1], len(iq_nuevos))
obs_idx_nuevos = np.arange(len(iq_nuevos_grupos))

with clone_model(modelo_3):
    pm.set_data(
        {"iq_madre": iq_nuevos_grupos, "hs_idx": secundaria_nuevos},
        coords={"obs_idx": obs_idx_nuevos},
    )
    preds_3 = pm.sample_posterior_predictive(
        idata_3, var_names=["mu", "score"], predictions=True, random_seed=SEMILLA
    )

In [ ]:
az.summary(
    preds_3,
    group="predictions",
    var_names="mu",
    kind="stats",
    ci_kind="eti",
    ci_prob=0.95,
    round_to=1,
)

Para resumir las medias usamos `az.summary()`. También podemos usarlo con cantidades derivadas fuera del modelo, reuniéndolas en un `Dataset` de xarray. Para las probabilidades y las diferencias que siguen vamos a extraer las muestras y trabajar con NumPy.

`az.extract()` conserva el orden de las seis predicciones. Las primeras tres corresponden a madres que no terminaron la secundaria y las siguientes tres a las que sí, siempre para IQ de 80, 100 y 120. Cada fila contiene una misma extracción del _posterior_ para todos esos casos.


In [ ]:
score_preds_3 = az.extract(preds_3, group="predictions", var_names="score")
score_preds_3 = score_preds_3.transpose("sample", "obs_idx").to_numpy()

In [ ]:
umbral_score = 110
(score_preds_3 > umbral_score).mean(axis=0)

Podemos calcular también la diferencia de medias entre grupos para cada IQ. Separamos las primeras tres columnas de las últimas tres y restamos dentro de cada fila, para conservar el emparejamiento entre las extracciones de los coeficientes.


In [ ]:
mu_preds_3 = az.extract(preds_3, group="predictions", var_names="mu")
mu_preds_3 = mu_preds_3.transpose("sample", "obs_idx").to_numpy()

mu_no_termino = mu_preds_3[:, : len(iq_nuevos)]
mu_termino = mu_preds_3[:, len(iq_nuevos) :]
diferencias_3 = mu_termino - mu_no_termino

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True, layout="constrained")

axes[0].hist(diferencias_3[:, 0], bins=40, density=True)
axes[1].hist(diferencias_3[:, 1], bins=40, density=True)
axes[2].hist(diferencias_3[:, 2], bins=40, density=True);

axes[0].set(
    xlabel="Diferencia",
    ylabel="Densidad",
    title=r"$\mu(80, \text{Terminó}) - \mu(80, \text{No Terminó})$"
)
axes[1].set(
    xlabel="Diferencia",
    title=r"$\mu(100, \text{Terminó}) - \mu(100, \text{No Terminó})$"
)
axes[2].set(
    xlabel="Diferencia",
    title=r"$\mu(120, \text{Terminó}) - \mu(120, \text{No Terminó})$"
);

In [ ]:
diferencias_3.mean(axis=0)

In [ ]:
np.quantile(diferencias_3, [0.025, 0.975], axis=0)

In [ ]:
(diferencias_3 > 0).mean(axis=0)

Al permitir pendientes diferentes, también cambiamos la comparación entre las medias. Con rectas paralelas, la diferencia era la misma para cualquier IQ y la probabilidad _a posteriori_ de que fuera positiva quedaba cerca de 0.997. Ahora esa probabilidad es aproximadamente 0.9995 para IQ 80, 0.950 para IQ 100 y 0.285 para IQ 120.

La diferencia media pasa de unos 10.5 puntos para IQ 80 a 3.9 para IQ 100 y -2.6 para IQ 120. Para los dos últimos IQ, los IC del 95% incluyen diferencias positivas y negativas. En particular, para IQ 120 el intervalo va aproximadamente de -11 a 6 puntos. Todavía hay bastante incertidumbre sobre esa comparación.

**¿Tenemos motivos para dejar que las pendientes cambien según la secundaria materna?** La interacción permite que la diferencia entre grupos se reduzca y llegue a cambiar de signo al aumentar el IQ. Esa es una característica que necesitamos evaluar con los datos, especialmente donde hay pocas madres que no terminaron la secundaria. Más adelante vamos a mirar si esta flexibilidad aporta una mejora predictiva suficiente como para conservarla.

### Diagnósticos del ajuste y comparación predictiva

Generemos también réplicas con el modelo de interceptos y pendientes diferentes, usando las covariables observadas. Después vamos a comparar los modelos de regresión para ver qué aspectos de los datos reproduce cada uno.

In [ ]:
with modelo_3:
    pm.sample_posterior_predictive(
        idata_3, var_names=["score"], extend_inferencedata=True, random_seed=SEMILLA
    )

In [ ]:
az.plot_ppc_dist(idata_3, var_names="score", num_samples=50);

En la comparación global todavía podemos ver una concentración de puntuaciones y una cola de valores bajos que las réplicas no reproducen del todo bien. Si bien incorporar covariables mejora algunos aspectos del ajuste, seguimos usando un modelo observacional normal con desvío constante.

Además, al mirar todas las puntuaciones juntas podríamos pasar por alto diferencias entre los grupos. Veamos entonces si los modelos reproducen la diferencia de medias que observamos entre los niños cuyas madres terminaron la secundaria y los del otro grupo.

Para cada conjunto replicado calculamos esa diferencia y después la comparamos con la observada. Acá usamos los IQ que tiene cada grupo en la muestra, por lo que se trata de una diferencia sin ajustar por IQ. Esto es distinto de las comparaciones a igual IQ que hicimos antes, y nos sirve para evaluar si el modelo reproduce este aspecto de los datos.

In [ ]:
diferencia_observada = score[secundaria_idx == 1].mean() - score[secundaria_idx == 0].mean()
replicas_modelos = [idata_1, idata_2, idata_3]
titulos_modelos = [
    "Intercepto y pendiente comunes",
    "Intercepto variable",
    "Intercepto y pendiente variables",
]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, layout="constrained")
for ax, idata_modelo, titulo in zip(axes, replicas_modelos, titulos_modelos):
    replicas = idata_modelo["posterior_predictive"]["score"]
    media_termino = replicas.isel(obs_idx=secundaria_idx == 1).mean("obs_idx")
    media_no_termino = replicas.isel(obs_idx=secundaria_idx == 0).mean("obs_idx")
    diferencia_replicada = (media_termino - media_no_termino).to_numpy().ravel()
    ax.hist(diferencia_replicada, bins=30, density=True, alpha=0.4)
    ax.axvline(diferencia_observada, ls="--", label="Observada", color="C1")
    ax.set(title=titulo, xlabel="Diferencia de medias (puntos)")
axes[0].set_ylabel("Densidad")
axes[0].legend();

También podemos comparar los residuos contra IQ por separado para las madres que terminaron la secundaria y las que no. Cada fila corresponde a un grupo y cada columna a un modelo. Usamos las mismas escalas para buscar desplazamientos persistentes respecto de cero, cambios en la dispersión y observaciones alejadas que merezcan una revisión.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True, layout="constrained")

for columna, idata_modelo, titulo in zip(range(3), replicas_modelos, titulos_modelos):
    media_ajustada = idata_modelo["posterior"]["mu"].mean(("chain", "draw")).to_numpy()
    residuos = score - media_ajustada
    for fila, grupo, etiqueta in zip(range(2), grupos, grupos_secundaria):
        ax = axes[fila, columna]
        seleccion = secundaria_idx == grupo
        ax.scatter(iq_madre[seleccion], residuos[seleccion], s=12, alpha=0.5)
        ax.axhline(0, color="0.4", ls="--")
        ax.set(
            title=f"{titulo}\n{etiqueta}",
            xlabel="IQ de la madre",
            ylabel="Residuo (puntos)",
        )

#### ¿Cuál predice mejor?

Construimos varios modelos para describir las puntuaciones y hacer predicciones. Ahora podemos preguntarnos cuál de ellos predice mejor la puntuación de un niño que no usamos para ajustarlo.

Una forma de evaluarlo sería dejar una observación afuera, ajustar el modelo con las restantes y ver cuánta densidad predictiva le asigna a la puntuación que dejamos afuera. Si repetimos el proceso para cada observación y sumamos los logaritmos de esas densidades, obtenemos una medida de capacidad predictiva mediante validación cruzada _leave-one-out_ (LOO).

$$
\operatorname{elpd}_{\mathrm{LOO}}
=\sum_{i=1}^n\log p(y_i\mid\boldsymbol y_{-i},\boldsymbol x).
$$

En este caso dejamos afuera la puntuación del niño y conservamos sus covariables conocidas. Así evaluamos la predicción de nuevos niños de una población similar para los que contamos con esa información. Cuanto mayor sea el valor obtenido, mejor es la capacidad predictiva según este criterio.

Ajustar cada modelo 434 veces sería muy costoso. Por eso vamos a usar PSIS-LOO, que aproxima esa cuenta a partir de las muestras del _posterior_ que ya tenemos. ArviZ necesita un grupo `log_likelihood` con la logdensidad de cada observación para cada extracción del _posterior_, que podemos agregar con `pm.compute_log_likelihood()` sin repetir MCMC.

La aproximación también tiene sus diagnósticos. En la salida vamos a ver los valores de Pareto $k$, que permiten identificar observaciones para las que puede ser poco confiable. Si aparece una advertencia, corresponde revisar ese problema antes de interpretar la comparación.

In [ ]:
with modelo_0:
    pm.compute_log_likelihood(idata_0)

In [ ]:
with modelo_1:
    pm.compute_log_likelihood(idata_1)

In [ ]:
with modelo_2:
    pm.compute_log_likelihood(idata_2)

In [ ]:
with modelo_3:
    pm.compute_log_likelihood(idata_3)

In [ ]:
idata_3["log_likelihood"]

Veamos el resultado de LOO para el modelo sin covariables.

In [ ]:
loo_0 = az.loo(idata_0, pointwise=True)
loo_0

Veamos el resultado de LOO para el modelo con IQ materno e intercepto y pendiente comunes a ambos grupos.

In [ ]:
loo_1 = az.loo(idata_1, pointwise=True)
loo_1

Veamos el resultado de LOO para el modelo con intercepto variable por grupo y pendiente común.

In [ ]:
loo_2 = az.loo(idata_2)
loo_2

Veamos el resultado de LOO para el modelo con intercepto y pendiente variables por grupo.

In [ ]:
loo_3 = az.loo(idata_3)
loo_3

In [ ]:
resultados_loo = {
    "Sin covariables": loo_0,
    "Intercepto y pendiente comunes": loo_1,
    "Intercepto variable": loo_2,
    "Intercepto y pendiente variables": loo_3,
}
comparacion = az.compare(resultados_loo, round_to=2)
comparacion

La tabla ordena los modelos según su capacidad predictiva estimada, que aparece en la columna `elpd`. La columna `se` contiene el error estándar de esa estimación.

Para compararlos podemos mirar `elpd_diff`, la diferencia respecto del que encabeza la tabla. En esta versión de ArviZ, los valores negativos indican menor capacidad predictiva. `dse` es el error estándar de esa diferencia y nos ayuda a evaluar su incertidumbre.

En el modelo con intercepto variable aparece `|elpd_diff| < 4`. Su diferencia respecto del modelo con interacción es de aproximadamente -2.96, con un error estándar cercano a 1.94. ArviZ usa esa marca para señalar modelos con predicciones similares y advertir que la aproximación normal usada para calcular `p_worse` puede estar mal calibrada. El valor 4 es un umbral orientativo y necesitamos leerlo junto con la incertidumbre y las predicciones que nos interesan.

In [ ]:
az.plot_compare(comparacion);

Los modelos que incorporan IQ predicen mejor que el modelo sin covariables. Agregar pendientes distintas produce una mejora adicional pequeña y cambia bastante algunas respuestas, como la probabilidad de que la diferencia entre medias sea positiva para IQ 120. Por eso conviene preguntarnos si necesitamos esa flexibilidad para el problema que estamos estudiando.

Para continuar este ejemplo, podemos quedarnos con interceptos diferentes y una pendiente común por su sencillez. La mejora predictiva de la interacción es modesta y su interpretación requiere sostener que la asociación con el IQ cambia según el grupo. Los datos permiten explorar esa posibilidad, pero la comparación no obliga a elegirla. La interacción queda como una forma de evaluar cuánto dependen nuestras respuestas de esa decisión. Un cambio de signo inesperado nos invita a revisar las rectas y los datos que las sostienen, y por sí solo no demuestra que el modelo esté mal.

También es útil mirar qué pasó con el _posterior_ de $\sigma$. Sin covariables se concentra alrededor de 20 puntos. Al agregar IQ baja a unos 18.3, y después cambia muy poco: aproximadamente 18.1 con interceptos distintos y 18.0 con la interacción. En los tres modelos de regresión, el desvío del _posterior_ de $\sigma$ ronda 0.6 puntos.

In [ ]:
resultados_sigma = [idata_0, idata_1, idata_2, idata_3]
etiquetas_sigma = [
    "Sin covariables",
    "Intercepto y pendiente comunes",
    "Intercepto variable",
    "Intercepto y pendiente variables",
]
muestras_sigma = [idata["posterior"]["sigma"].to_numpy().flatten() for idata in resultados_sigma]

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
for muestras, etiqueta in zip(muestras_sigma, etiquetas_sigma):
    grilla_sigma, densidad_sigma, _ = az.kde(muestras)
    ax.plot(grilla_sigma, densidad_sigma, linewidth=2, label=etiqueta)
ax.set(xlabel=r"$\sigma$ (puntos)", ylabel="Densidad", title="Distribución a posteriori del desvío residual", ylim=(0, 0.9))
ax.legend(loc="upper right");

Esta comparación distingue dos cosas. Al incorporar IQ reducimos algo la dispersión de las puntuaciones alrededor de la media que predice el modelo. La incertidumbre sobre el valor de $\sigma$ sigue siendo parecida entre los modelos de regresión. Podemos conocer bastante bien ese desvío y, al mismo tiempo, tener mucha incertidumbre al predecir la puntuación de un niño. Con $\sigma\approx18$, incluso una normal con parámetros conocidos tendría un intervalo central del 95% de unos $\pm35$ puntos alrededor de su media.

El IQ de la madre está asociado positivamente con la puntuación del niño, pero conocerlo todavía deja una dispersión grande entre puntuaciones individuales. El modelo probablemente deja afuera otros factores asociados a esas puntuaciones. La variación que queda también puede incluir error de medición y limitaciones de los supuestos que usamos. Agregar secundaria o permitir pendientes diferentes apenas reduce esa dispersión.

<a id="que-hacer-diagnosticos"></a>

## ¿Qué hacer cuando los diagnósticos dan mal?

La [Sección 4.6 de _Exploratory Analysis of Bayesian Models_](https://arviz-devs.github.io/EABM/Chapters/MCMC_diagnostics.html#what-to-do-when-the-diagnoses-are-wrong) propone varias alternativas según el problema observado.

- **Leer las advertencias del _sampler_.** Pueden orientar la revisión del ajuste.
- **Aumentar las iteraciones o la adaptación.** Más `draws` o `tune` pueden ayudar cuando las dificultades son menores.
- **Reparametrizar.** Como vimos con las ocho escuelas, otra implementación del mismo modelo puede mejorar la exploración.
- **Revisar el modelo y los _priors_.** Las dificultades computacionales pueden señalar supuestos poco adecuados. Conviene incorporar información previa justificada por el problema, evitando elegir _priors_ solamente para silenciar advertencias.
- **Aumentar `target_accept` si hay divergencias.** Por ejemplo, probar `pm.sample(target_accept=0.95)`. Si persisten cerca de 0.99, conviene revisar la parametrización y el modelo.
- **Considerar otro método de muestreo.** La elección depende de las características del modelo.

Después de cada cambio, hay que volver a mirar los diagnósticos en conjunto antes de interpretar el _posterior_.

## Sección adicional

Retomemos el primer modelo de regresión para explorar otra forma de expresar sus coeficientes.


<a id="dependencia-intercepto-pendiente"></a>

### Dependencia entre el intercepto y la pendiente

En este primer modelo usamos el IQ en su escala original. Como los valores observados están lejos de cero, un aumento de la pendiente puede compensarse con una disminución del intercepto y producir medias parecidas en el rango de los datos. Veamos la distribución conjunta de estos coeficientes y su correlación _a posteriori_.


In [ ]:
az.plot_pair(idata_1, var_names=["beta_0", "beta_1"]);

In [ ]:
xr.corr(
    idata_1["posterior"]["beta_0"], idata_1["posterior"]["beta_1"], dim=("chain", "draw")
).item()

La nube alargada y la correlación negativa muestran que los coeficientes están fuertemente asociados en el _posterior_. Esta geometría puede dificultar la exploración de NUTS y reducir la cantidad de muestras efectivas por iteración. Por eso conviene mirar los ESS de los coeficientes junto con las trazas, $\hat R$ y las divergencias.

Una correlación alta por sí sola no implica que el muestreo haya fallado. Si $\hat R$ queda cerca de 1 y no hay divergencias, todavía necesitamos evaluar si el ESS y el error de Monte Carlo alcanzan para nuestras consultas. Además, el intercepto corresponde a IQ cero, por lo que su interpretación requiere extrapolar muy lejos de los datos.


### Centrar el IQ

¿Qué pasaría si, en vez de usar el IQ en su escala original, trabajáramos con $x_i^*=x_i-100$? Podemos reescribir la recta para ver qué cambia.

$$
\beta_0+\beta_1x_i
= (\beta_0+100\beta_1)+\beta_1(x_i-100).
$$

La pendiente sigue siendo la misma, pero el nuevo intercepto es $\beta_0^*=\beta_0+100\beta_1$. Ahora representa la puntuación media para madres con IQ 100, que sí es un valor cercano a las observaciones.

Podemos transformar las muestras que ya tenemos y comprobar que las medias condicionales que obtenemos son las mismas.

In [ ]:
posterior_1 = idata_1["posterior"]
iq_referencia = 100
iq_centrado = iq_madre - iq_referencia
beta_0_centrado = posterior_1["beta_0"] + iq_referencia * posterior_1["beta_1"]

mu_original = (
    posterior_1["beta_0"].to_numpy()[:, :, None]
    + posterior_1["beta_1"].to_numpy()[:, :, None] * iq_madre
)
mu_centrado = (
    beta_0_centrado.to_numpy()[:, :, None]
    + posterior_1["beta_1"].to_numpy()[:, :, None] * iq_centrado
)

In [ ]:
np.allclose(mu_original, mu_centrado)

El intervalo del nuevo intercepto describe la puntuación media para madres con IQ 100.

In [ ]:
beta_0_centrado.quantile([0.025, 0.975])

In [ ]:
posterior_centrado = xr.Dataset(
    {"beta_0_centrado": beta_0_centrado, "beta_1": posterior_1["beta_1"]}
)
az.plot_pair(posterior_centrado, var_names=["beta_0_centrado", "beta_1"]);

Además de tener una interpretación más útil, el nuevo intercepto presenta una correlación menor con la pendiente. Podemos comparar este gráfico con el del primer modelo, donde usamos el IQ en su escala original. Como obtuvimos estos valores transformando las muestras disponibles, seguimos trabajando con el mismo modelo ajustado. Esta transformación facilita la interpretación, pero conserva las dificultades de exploración que haya tenido el muestreo original.

Si quisiéramos escribir y ajustar el modelo con el IQ centrado desde el comienzo, también tendríamos que pensar el _prior_ de ese intercepto. La transformación de los _priors_ originales induce dependencia entre $\beta_0^*$ y $\beta_1$. Si en su lugar eligiéramos nuevos _priors_ independientes, estaríamos cambiando los supuestos del modelo.
